# 리포트 03 — 조명원 — 세 파형과 그 대가

> ### ❓ 이 편이 답하는 질문
> **패시브 레이더가 빌려 쓸 수 있는 상시 신호는 무엇이고, 그 선택은 몇 dB 의 대가를 치르는가?**

### 결론
1. 패시브가 상관에 쓸 수 있는 것은 셀이 **늘 켜 두는 기준신호**뿐이다 — LTE=CRS(18.0 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.ref_bw_mhz⟩), 5G=SSB(7.2 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.ref_bw_mhz⟩), WiFi=VHT-LTF(76.6 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.ref_bw_mhz⟩).
2. **5G 는 이중고**다 — 기준신호가 좁아 ΔR_b = 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ (LTE 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩), 드물어서 PRF 50 Hz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.prf_hz⟩ → 무모호 속도 1.07 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.vmax_ms⟩ (LTE 40.7 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.vmax_ms⟩).
3. ⭐ 이 편의 수치는 **표적 σ 와 무관**하다 — 파형·반송파·기하만으로 정해지므로 02편의 RCS 감사가 무엇을 바꾸든 움직이지 않는다(§0).
4. 조명원 선택의 대가는 dB 로 닫힌다 — 점유 18.0 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩, 반송파 λ² -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.span_db⟩(밴드 양끝), WiFi 패킷 듀티 -12.84 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.packet_duty_db⟩.
5. 파형은 Sionna PHY 독립 변조기와 상관 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ 로 일치하고, 우리가 그리는 모호함수는 검출기 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ 안에서 같다.

### ✅ 주장하는 것 / ❌ 주장하지 않는 것

| ✅ 이 편이 주장하는 것 | ❌ 이 편이 주장하지 않는 것 |
|---|---|
| 세 조명원의 **B_ref · ΔR_b · PRF · 듀티 · 모호함수** — 자원격자에서 잰 정확한 양 | 기준신호 **배치 좌표**(CRS·SSB·VHT-LTF)의 독립 검증 — Sionna 에 생성기가 없다(§3) |
| **상시 신호만 쓰는 체제에서 LTE CRS > 5G SSB** — 거리·속도 두 축 모두 | **full-waveform capture** 체제 — 이 편은 상시 신호만 쓰는 기본선이다(§1) |
| OFDM **변조 단계**가 Sionna PHY 와 일치한다 — 상관·NMSE 로 정량화(§3) | 검출확률·CFAR·Pfa — 04편 소관. 이 편은 σ 를 곱하기 **전** 단계다 |
| §4 의 모호함수는 **검출기가 실제로 내는 응답**이다 — 편차 상한을 명시(§4) | 표적 밝기 σ 와 그 밴드 의존성 — 02편 소관 |
|  | WiFi 의 상시성 — PRF 는 혼잡 AP 대표값이지 보장된 값이 아니다(마지막 절) |

### 필요한 사전지식

없음. 이 편부터 읽어도 된다.

### 재현

```bash
cd /home/yunjung/workspace/sionna2
# ① 파형 제원 · 자원격자 · Sionna 교차대조 (그림 4장 포함)
~/.venvs/py312/bin/python src/viz_report2.py
# ② 모호함수 — 검출기와 같은 커널
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
# ③ 링크버짓 규약 상수(듀티 · CPI · CFAR)
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/report4_fixups.py
# ④ 이 편의 파생 원장 + 그림 3장 + 노트북
PYTHONPATH=src ~/.venvs/py312/bin/python src/make_report03_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/verify_ambiguity.json`, `outputs/report4_fixups.json`, `outputs/report5_results.json`, `outputs/report03_illuminators.json` |
| 소요 | ① 3412 s ⟨outputs/report2_waveform_rcs.json : meta.runtime_s⟩ (대부분은 같은 스크립트의 RCS 스윕이고 파형 부분은 초 단위) · ③ 556 s ⟨outputs/report4_fixups.json : _meta.runtime_s⟩ · ② GPU 1장 수 분 · ④ 초 단위 |
| 비고 | outputs/report5_results.json 는 검출 몬테카를로가 이미 남긴 것이다 — 이 편은 그중 `A_occupancy` 만 인용한다(재실행 불필요). |

---

## §0. 이 편의 수치는 무엇에 의존하는가

조명원 쪽 양들은 **파형·반송파·관측시간**만으로 정해진다. 표적의 σ 를 곱하기 **전** 단계라서, σ 를 둘러싼 논쟁이 어떻게 끝나든 아래 값은 바뀌지 않는다. 인수인계 시 **가장 먼저 믿어도 되는 층**이다.

| 양 | 무엇이 정하나 | σ 에 의존? | 이 편의 절 |
|---|---|---|---|
| 기준신호 대역 $B_{ref}$ → $\Delta R_b$ | 자원격자 | 없음 | §1 |
| PRF → 무모호 속도 | 기준신호 반복주기 | 없음 | §1 |
| 점유 · 듀티 · λ² · √(B/f_s) | 파형 · 반송파 · 표본율 | 없음 | §2 |
| 모호함수(주엽·부엽·레플리카) | 기준신호 파형 + 창 | 없음 | §4 |
| 검출확률 $P_d$ · CFAR 문턱 | 위 전부 **× σ × 기하** | **있음** | 04 · 05편 |

## §1. 세 조명원 — 무엇이 늘 켜져 있는가

패시브 수신기는 두 조건을 **동시에** 만족하는 신호에만 상관을 걸 수 있다. **① 내용을 미리 안다** — 데이터는 매 순간 바뀌므로 못 쓴다. **② 아무 셀이나 늘 켠다** — 조건부로 켜지는 신호는 하필 그때 없으면 표적을 놓친다.

두 조건을 다 만족하는 신호는 표준마다 **하나씩**이다. 격자에서 잰 제원은 다음과 같다.

| 표준 | 상시 기준신호 | 반송파 | 채널 점유대역 | $B_{ref}$ | $\Delta R_b=c/B_{ref}$ |
|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 5.21 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.carrier_ghz⟩ | 75.6 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.chan_bw_mhz⟩ | 76.6 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.ref_bw_mhz⟩ | 3.9 m ⟨outputs/report2_waveform_rcs.json : reference.G1.wifi.dR_m⟩ |
| LTE Rel-9 | CRS | 1.843 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.carrier_ghz⟩ | 18.0 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.chan_bw_mhz⟩ | 18.0 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.ref_bw_mhz⟩ | 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ |
| 5G NR Rel-16 | SSB | 3.50 GHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.carrier_ghz⟩ | 98.3 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.chan_bw_mhz⟩ | 7.2 MHz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.ref_bw_mhz⟩ | 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ |

$B_{ref}$ 는 기준신호가 차지한 부반송파의 **양끝 span** 이다(`src/waveforms.py:237`) — 안쪽 널 톤을 포함하므로 WiFi 처럼 span 이 점유대역보다 넓을 수 있다. 격자는 `src/waveforms.py:258`(WiFi) · `:313`(LTE) · `:370`(5G), $\Delta R_b$ 는 `src/waveforms.py:144`.

![resource grid](outputs/figures/report2_resource_grid.png)

**그림 1.** 유휴 셀이 실제로 켜는 칸은 어디이고, 그중 패시브가 상관에 쓸 수 있는 것은 무엇인가?

### §1.1 5G 의 이중고 — 좁고, 드물다

**PRS 는 상시 신호가 아니다.** 넓은 대역을 켜지만 **측위 세션이 설정돼야** 켜지는 옵션이고, 남의 셀을 빌리는 패시브 수신기는 그것이 켜져 있다고 가정할 수 없다. 따라서 5G 의 기본선은 SSB 이고, PRS 를 켠 수치는 **낙관적 상한**으로만 읽는다.

| 축 | 정하는 것 | LTE CRS | 5G SSB | 격차 |
|---|---|---|---|---|
| 거리 $\Delta R_b$ | $B_{ref}$ | 16.7 m ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.dR_m⟩ | 41.6 m ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.dR_m⟩ | 2.5배 거칢 |
| 속도 $v_{max}$ (모노 등가) | PRF | 40.7 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.lte.vmax_ms⟩ | 1.07 m/s ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.vmax_ms⟩ | 38배 빠름 |

SSB 는 PRF 50 Hz ⟨outputs/report2_waveform_rcs.json : reference.G1.nr.prf_hz⟩ — **걷는 속도의 드론도 도플러가 접힌다**(§4.3). 통신 세대가 최신이라고 조명원으로 좋은 것은 아니다.

> ⚠ 반대로 **WiFi 가 이기는 것도 조건부**다. 위 PRF 는 혼잡 AP 값이고, 비콘만 내보내는 유휴 AP 면 속도축에서 5G 보다 나쁘다(그림 2 의 Caveat 2, §5).

![reference signal budget](outputs/figures/report2_ref_signal.png)

**그림 2.** 기준신호의 넓이와 반복이 거리·속도 눈금을 각각 얼마로 정하는가?

## §2. 대가 원장 — 전부 정확한 양

조명원 선택이 만드는 dB 격차를 한 표에 모은다. 모든 항목이 **두 양의 비**라서 표적 σ 가 상쇄된다.

| 항목 | 값 | 무엇의 비인가 |
|---|---|---|
| 점유 대가 (5G · 상시 vs 풀로드) | 18.0 dB ⟨outputs/report03_illuminators.json : occupancy_cost.value_db⟩ | 같은 표적·같은 기하에서 $P_d$ 0.5 ⟨outputs/report03_illuminators.json : occupancy_cost.pd_threshold⟩ 를 넘기는 EIRP 차 |
| 기준신호 에너지 격차 (같은 쌍) | 12.70 dB ⟨outputs/report03_illuminators.json : ref_energy_gap_G1_to_G3_db.nr⟩ | $E_{ref}$(G3) / $E_{ref}$(G1) — 상관에 쓸 수 있는 에너지만 |
| 반송파 λ² (LTE→WiFi) | -9.03 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_wifi_db⟩ | $20\log_{10}(\lambda/\lambda_{ref})$ — EIRP·수신이득 고정 |
| 반송파 λ² (LTE→5G) | -5.57 dB ⟨outputs/report03_illuminators.json : lambda2.lte_to_nr_db⟩ | 위와 같음 |
| WiFi 파일럿 / 총 송신 에너지 | -11.27 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.pilot_over_tx_energy_db⟩ | G3 격자에서 상관에 쓸 수 있는 몫 |
| WiFi 패킷 듀티 | -12.84 dB ⟨outputs/report4_fixups.json : F4_linkbudget.wifi_pilot_fraction.packet_duty_db⟩ | 패킷이 공중에 있는 시간 비율 |
| CPI 규약 격차 | 3.01 dB ⟨outputs/report4_fixups.json : F4_linkbudget.cpi_asymmetry.span_db⟩ | 같은 프레임 수 M 이 5G 에 주는 관측시간이 절반 |

> ⚠ **부호는 원본 JSON 그대로다.** 손해가 양수로 적히는 항목(점유·기준신호 에너지·CPI)과 음수로 적히는 항목(λ²·WiFi 두 항목)이 섞여 있다 — 그림 4 는 전부 '음수 = 손해'로 부호를 맞춰 다시 그린 것이다.

> ⚠ **점유 대가는 '점유만'의 값이 아니다.** G1→G3 은 기준신호 대역도 7.2 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G1_mhz⟩ → 98.3 MHz ⟨outputs/report03_illuminators.json : occupancy_cost.ref_bw_G3_mhz⟩ 로 함께 넓어진다. 두 효과를 분리하려면 대역을 고정한 채 점유만 바꾼 스윕이 따로 필요하다.

> ⚠ 앞의 두 WiFi 항목은 **서로 다른 양**이다. 하나는 에너지 비, 하나는 시간 비다 — 곱해서 한 숫자로 만들면 안 된다.

> ⚠ **λ² 는 전제가 붙는다.** EIRP 가 고정이고 수신 안테나 **이득**이 고정일 때만 성립한다(`src/freespace_link.py:371`). 수신 **개구면적**을 고정하면 부호가 뒤집힌다.

> ⚠ **CPI 규약 격차는 통제해야 할 변수다.** 파형 비교에서 관측시간이 다르면 그 차이가 파형의 우열로 오독된다 — 04·05편은 관측시간을 맞춘 뒤에 비교한다.

![occupancy](outputs/figures/report2_occupancy.png)

**그림 3.** 셀이 데이터로 바빠지면 패시브의 거리분해능도 같이 좋아지는가?

![cost ledger](outputs/figures/report03_cost_ledger.png)

**그림 4.** 조명원 선택이 무는 대가는 항목별로 몇 dB 인가?

### §2.1 규약 — 여기서 틀리면 2배가 틀린다

이 프로젝트의 거리축은 **바이스태틱 거리합** $R_b=R_1+R_2-L$ 이므로 분해능은 $\Delta R_b=c/B_{ref}$ 다. 모노스태틱 교과서 값 $c/2B$ 는 그 절반이다 — 비 2 ⟨outputs/report4_fixups.json : F3_ambiguity.resolution_convention_conflict.rows[0].factor⟩배. 섞어 쓰면 분해능을 그만큼 낙관하게 된다.

잡음대역 정규화 √(B/f_s) 도 같은 성격의 규약이다 — 선언한 대역 B 와 표본율 f_s 가 다르면 주입 진폭을 그만큼 낮춰야 매치드필터 출력 SNR 이 파형 간 공정해진다(`benchmark/run_min_cell.py:131`).

| 파형 | $B/f_s$ | $\Delta R_b=c/B_{ref}$ | 모노 등가 $c/2B$ |
|---|---|---|---|
| WiFi 80MHz | 0.9453 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[0].b_over_fs⟩ | 3.92 m ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_theory_m⟩ | 1.96 m ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_mono_theory_m⟩ |
| LTE 20MHz | 0.5859 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[1].b_over_fs⟩ | 16.67 m ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.dR_theory_m⟩ | 8.33 m ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.dR_mono_theory_m⟩ |
| 5G 100MHz | 0.7998 ⟨outputs/report4_fixups.json : F4_linkbudget.straddle.rows[2].b_over_fs⟩ | 41.64 m ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_theory_m⟩ | 20.82 m ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_mono_theory_m⟩ |

In [ ]:
# §2 원장을 JSON 에서 그대로 읽어 찍는다 — 본문 숫자에 하드코딩이 없음을 확인하는 셀.
import json
L = json.load(open('outputs/report03_illuminators.json'))
print('점유 대가      ', f"{L['occupancy_cost']['value_db']:+.1f} dB",
      f"(G1 {L['occupancy_cost']['eirp_G1_dbm']:+.0f} dBm vs "
      f"G3 {L['occupancy_cost']['eirp_G3_dbm']:+.0f} dBm, "
      f"격자 {L['occupancy_cost']['eirp_grid_step_db']:.0f} dB)")
for k in ('wifi', 'lte', 'nr'):
    g1, g3 = L['grids']['G1'][k], L['grids']['G3'][k]
    print(f"{k:5s} G1 ref={g1['ref_name']:8s} B_ref={g1['ref_bw_hz']/1e6:6.2f} MHz "
          f"dRb={g1['drb_m']:6.2f} m  E_ref/E_tx={g1['e_ref_over_tx_db']:+6.2f} dB"
          f"   | G1->G3 기준신호 에너지 "
          f"{L['ref_energy_gap_G1_to_G3_db'][k]:+5.2f} dB")

## §3. 파형이 규격대로인가 — Sionna PHY 로 채점

Sionna PHY 는 5G NR 뉴머롤로지(`nr.CarrierConfig`, `TS 38.211` 구현)와 OFDM 변조기는 주지만, **WiFi·LTE 파형도 상시 기준신호(CRS·SSB·VHT-LTF)의 격자 배치도 주지 않는다.** 그래서 격자는 규격서를 읽어 `src/waveforms.py` 의 표준별 생성기(§1 의 줄번호)에 세우고, **격자를 신호로 바꾸는 단계**만 독립 변조기 `sionna.phy.ofdm.OFDMModulator` 로 채점한다(`src/viz_report2.py:387`).

| 이 대조가 **확인해 주는 것** | 이 대조가 **못 하는 것** |
|---|---|
| IFFT 규약 — fftshift 방향·정규화 | 가드밴드·DC 널 배치 (격자는 우리가 건넨 입력) |
| CP 복사와 심볼별 이어붙이기 순서 | CP 길이 **값**이 3GPP 표와 맞는가 |
| 두 독립 구현이 같은 시간파형을 내는가 | 파일럿 **좌표** (Sionna 에 생성기가 없다) |

![crosscheck](outputs/figures/report2_crosscheck.png)

**그림 5.** 같은 자원격자를 두 변조기에 넣으면 같은 시간파형이 나오는가?

세 파형 모두 상관이 소수 넷째 자리까지 1 이고, 남은 차이는 float32 반올림 바닥이다. 오른쪽 열은 **대조의 분해력 시험**이다 — 재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 두 번째 심볼부터 시간축이 어긋나 상관이 무너진다. CP 가 심볼마다 같은 WiFi 는 그대로다.

| 표준 | 표본 수 | $f_s$ | 상관 | NMSE | CP 앞머리 | CP 배열을 안 넘기면 |
|---|---|---|---|---|---|---|
| WiFi 802.11ac | 4160 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.n⟩ | 80.00 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.corr⟩ | -138.3 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.nmse_db⟩ | `[64]` | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.wifi.corr_bug⟩ |
| LTE Rel-9 | 30720 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.n⟩ | 30.72 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr⟩ | -135.6 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.nmse_db⟩ | `[160, 144, 144, 144]` | 0.0634 ⟨outputs/report2_waveform_rcs.json : crosscheck.lte.corr_bug⟩ |
| 5G NR Rel-16 | 61440 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.n⟩ | 122.88 MHz ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.fs_mhz⟩ | 1.0000 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr⟩ | -135.2 dB ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.nmse_db⟩ | `[352, 288, 288, 288]` | 0.0451 ⟨outputs/report2_waveform_rcs.json : crosscheck.nr.corr_bug⟩ |

대조는 **G3(풀로드) 격자 하나**에서 돈다 — '세 파형 모두'는 그 조건 아래의 진술이다.

## §4. 모호함수 — 검출기가 실제로 보는 눈

모호함수 $\chi(\tau,f_d)$ 는 기준신호 하나가 거리-도플러 평면에 만드는 응답이다. 우리가 그리는 것은 별도 계산이 아니라 **검출기가 쓰는 것과 같은 커널**이고, 검출기의 거리도플러 출력과 최대 0.144 dB ⟨outputs/report03_illuminators.json : detector_af_max_err_db.value⟩ (6 ⟨outputs/report03_illuminators.json : detector_af_max_err_db.n_cases⟩개 경우, −45 dB 이상 셀) 안에서 같다. 코드: `benchmark/verify_ambiguity.py:150`, 검출기는 `src/passive_process.py:133`.

### §4.1 주엽 — 닫힌형이 맞는가

거리 주엽은 $c/B_{ref}$ 예측의 89% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dR_ratio⟩ ~ 94% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.dR_ratio⟩ 이고, 도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.dF_ratio⟩배 근처다 — 이 배수는 파형이 아니라 **slow-time Hann 창**이 정한다(`src/passive_process.py:142`).

![ambiguity mainlobe](outputs/figures/report03_af_mainlobe.png)

**그림 6.** 측정한 모호함수 주엽이 닫힌형 예측과 몇 % 안에서 맞는가?

### §4.2 부엽과 도플러 레플리카 — 검출에 무엇을 뜻하나

주엽 밖으로 새는 에너지는 두 가지로 나타난다. **부엽**은 강한 표적이 평면 다른 곳의 약한 표적을 덮는 정도이고, **±PRF 의 레플리카**는 무모호 속도를 넘은 표적이 되접혀 들어오는 세기다.

⚠ 이 표의 PRF 는 **검출기 프레임률**이다. 물리 SSB 주기 기준의 접힘은 §4.3 이 따로 다룬다(§5).

| 기준신호 | 2D 부엽 최대 | ±PRF 레플리카 | 프레임 내 시간점유 | 함의 |
|---|---|---|---|---|
| WiFi VHT-LTF | -14.3 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.psl_2d_db⟩ | -0.00 dB ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.doppler_replica_db⟩ | 0.4% ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.ref_time_duty⟩ | 레플리카가 **무손실** — 접힘이 그대로 산다 |
| LTE CRS | -5.3 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.psl_2d_db⟩ | -23.27 dB ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.doppler_replica_db⟩ | 42.9% ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.ref_time_duty⟩ | 레플리카는 죽지만 부엽이 가장 높다 |
| 5G SSB | -18.0 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.psl_2d_db⟩ | -1.05 dB ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.doppler_replica_db⟩ | 28.6% ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.ref_time_duty⟩ | 부엽은 낮으나 레플리카가 거의 무손실 |

레플리카를 정하는 것은 점유율이 아니라 **에너지가 프레임 안에 얼마나 퍼져 있는가**다 — CRS 처럼 프레임 전체에 흩어지면 위상이 상쇄되고, LTF·SSB 처럼 앞쪽에 뭉치면 상쇄가 없다.

![ambiguity sidelobe](outputs/figures/report03_af_sidelobe.png)

**그림 7.** 각 기준신호는 표적 에너지를 부엽과 도플러 레플리카에 얼마나 남기는가?

### §4.3 접힘 — 5G SSB 는 걷는 드론도 놓친다

SSB 의 물리 반복률은 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 다. 무모호 도플러가 ±25 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_unamb_phys_hz⟩ 뿐이라, 이 프로젝트의 기준 표적 속도에서 참 도플러 64.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_true_hz⟩ 가 14.0 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.fd_aliased_phys_hz⟩ 로 접힌다(`aliased` = 예 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.aliased⟩). 같은 조건에서 WiFi·LTE 는 접히지 않는다.

| 기준신호 | 물리 PRF | 무모호 속도 | 접히는가 |
|---|---|---|---|
| WiFi VHT-LTF | 1000 Hz ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.prf_physical_hz⟩ | 14.4 m/s ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.v_unamb_phys_ms⟩ | 아니오 ⟨outputs/verify_ambiguity.json : waveforms.wifi_G1.physical.aliased⟩ |
| LTE CRS | 1000 Hz ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.prf_physical_hz⟩ | 40.7 m/s ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.v_unamb_phys_ms⟩ | 아니오 ⟨outputs/verify_ambiguity.json : waveforms.lte_G1.physical.aliased⟩ |
| 5G SSB | 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ | 1.07 m/s ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.v_unamb_phys_ms⟩ | 예 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.aliased⟩ |

이것이 §1 이 말한 이중고의 두 번째 절반이다 — 5G 는 **좁아서** 거리를 못 가르고, **드물어서** 속도를 못 가른다.

## §5. 이 편의 한계

| 아직 안 되어 있는 것 | 다음 사람이 이어받을 지점 |
|---|---|
| 기준신호 **격자 좌표**(CRS·SSB·VHT-LTF)의 독립 검증이 없다 — Sionna 에 생성기가 없어 §3 대조는 변조 단계까지만 닿는다 | X410 으로 실제 셀을 캡처해 `src/waveforms.py` 의 격자와 대조 → 06편 측정 설계에 항목 추가 |
| **§4 의 모호함수는 검출기 프레임률로 계산된다** — 5G 는 프레임률 2000 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_model_hz⟩, 물리 SSB 반복은 50 Hz ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.prf_physical_hz⟩ 로 40 ⟨outputs/verify_ambiguity.json : waveforms.nr_G1.physical.ratio⟩배 차이다. §4.3 의 접힘 표만 물리 PRF 로 따로 냈다 | `benchmark/run_min_cell.py:74` 의 `frame_len()` 을 물리 SSB 주기로 확장해 `benchmark/verify_ambiguity.py:108` 이 같은 프레임을 쓰게 하고 §4 표 전체를 재계산 |
| 점유 대가는 EIRP 격자 6 dB ⟨outputs/report03_illuminators.json : occupancy_cost.eirp_grid_step_db⟩ 간격에서 읽은 값이고, 표적 mavic4pro ⟨outputs/report03_illuminators.json : occupancy_cost.drone⟩ · 시나리오 radial ⟨outputs/report03_illuminators.json : occupancy_cost.scen⟩ · 시행 60 ⟨outputs/report03_illuminators.json : occupancy_cost.n_trials⟩회 한 점에서만 쟀다 | `benchmark/run_matrix.py:300` 의 `eirps` 를 2 dB 간격으로 좁히고 §2 의 대역 고정 스윕을 추가해 '점유만'의 값을 분리 |
| **WiFi 의 상시성이 가장 약하다** — PRF 는 혼잡 AP 대표값이고, 비콘만 나오는 유휴 AP 는 훨씬 드물다. 문헌이 트래픽을 일부러 유발하는 이유다 | `src/waveforms.py:112` 의 `PILOT_RATE_HZ` 를 트래픽 시나리오 파라미터로 올리고 §1 표를 시나리오별로 |
| 이 편은 **상시 신호만** 쓰는 체제의 기본선이다 — 부하가 걸린 셀에서 파형 전체를 캡처하는 체제는 재지 않았다 | 05편 검출 비교에 'full-waveform capture' 축을 하나 더 세우고 두 체제를 섞어 인용하지 않기 |
| **λ² 항은 수신 안테나 이득 고정 전제**에 묶여 있다 — 개구면적 고정이면 부호가 뒤집힌다 | 06편 측정 설계에서 실제 수신 안테나를 확정한 뒤 `src/freespace_link.py:371` 전제를 재확인 |